# MLP — Baseline Deep Learning cho 4 chuỗi

Notebook này xây dựng mô hình **MLP ban đầu** để dự báo điện năng giờ kế tiếp (t+1), tạo kết quả mốc trước khi thử nghiệm cải tiến nâng cao.

## Mục tiêu
- Huấn luyện riêng trên 4 chuỗi `MT_234`, `MT_217`, `MT_162`, `MT_314`.
- So sánh với baseline truyền thống đã có.
- Giữ nguyên split theo thời gian, không rò rỉ tương lai.
- Dùng Validation để chọn epoch tốt nhất; Test chỉ dùng đánh giá cuối.

## Cấu hình ban đầu
- Lookback: 168 giờ
- Horizon: 1 giờ
- Loss: MSE; optimizer: Adam
- Early stopping: patience 10
- Metrics: MAE, RMSE, MASE

Đây là **baseline DL**, chưa phải mô hình nâng cao.

## 1. Import thư viện và đọc cấu hình project

Notebook sử dụng các hàm dữ liệu và metric hiện có trong `src/` để giữ quy trình nhất quán với baseline.

In [ ]:
from pathlib import Path
import copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import sys
from pathlib import Path

# Xác định thư mục gốc của dự án (nếu notebook nằm trong notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import ROOT_DIR, load_config
from src.data import load_hourly_data, split_by_time
from src.metrics import evaluate_forecast, calculate_d_train

SEED = 42
LOOKBACK = 168
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
MAX_EPOCHS = 100
PATIENCE = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

cfg = load_config()
CUSTOMERS = cfg["customers"]["hourly_file_columns"]

OUT_DIR = ROOT_DIR / "results" / "dl_baseline" / "mlp"
TABLE_DIR, PRED_DIR, FIG_DIR = OUT_DIR / "tables", OUT_DIR / "predictions", OUT_DIR / "figures"
MODEL_DIR = ROOT_DIR / "models" / "dl_baseline"
for folder in [TABLE_DIR, PRED_DIR, FIG_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Customers:", CUSTOMERS)


## 2. Load dữ liệu và chia Train / Validation / Test

Dữ liệu được chia theo mốc thời gian từ `configs/config.yaml`. Không dùng random split. Các tập phải có thứ tự thời gian Train trước Validation trước Test.

In [ ]:
df = load_hourly_data(cfg, customers=CUSTOMERS)
train_df, val_df, test_df = split_by_time(df, cfg)

print("Train:", len(train_df), train_df.index.min(), "->", train_df.index.max())
print("Validation:", len(val_df), val_df.index.min(), "->", val_df.index.max())
print("Test:", len(test_df), test_df.index.min(), "->", test_df.index.max())

assert train_df.index.max() < val_df.index.min()
assert val_df.index.max() < test_df.index.min()


## 3. Chuẩn hóa và tạo sliding window

`StandardScaler` được fit **chỉ trên Train**. Sau đó áp dụng scaler đó lên Validation và Test.

Với mỗi timestamp t:
- Input: 168 giờ trước đó `X[t-168:t]`
- Target: giá trị tại giờ t `y[t]`

Validation lấy lịch sử cuối Train làm context; Test lấy lịch sử cuối Validation làm context. Không dùng target Validation/Test để cập nhật model.

In [ ]:
def make_windows(values, lookback=LOOKBACK):
    values = np.asarray(values, dtype=np.float32).reshape(-1)
    X, y = [], []
    for t in range(lookback, len(values)):
        X.append(values[t-lookback:t])
        y.append(values[t])
    return np.asarray(X, dtype=np.float32).reshape(-1, lookback, 1), np.asarray(y, dtype=np.float32).reshape(-1, 1)

def make_eval_windows(context_values, eval_values, lookback=LOOKBACK):
    joined = np.concatenate([
        np.asarray(context_values, dtype=np.float32).reshape(-1)[-lookback:],
        np.asarray(eval_values, dtype=np.float32).reshape(-1)
    ])
    return make_windows(joined, lookback)


## 4. Định nghĩa MLP

MLP xem 168 giá trị lịch sử như một vector đầu vào và truyền qua các lớp Dense.

Kiến trúc baseline: `168 → 128 → 64 → 1`, với ReLU và Dropout 0.1. Cấu hình này được dùng làm mốc, chưa tinh chỉnh nâng cao.

In [ ]:
class ForecastModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LOOKBACK, 128),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # [batch, lookback, 1] -> [batch, lookback]
        return self.net(x.squeeze(-1))


## 5. Huấn luyện và Early Stopping

- Loss: MSE trên dữ liệu đã chuẩn hóa.
- Optimizer: Adam.
- Mỗi epoch đánh giá Validation loss.
- Lưu trọng số tại epoch có Validation loss thấp nhất.
- Dừng sớm nếu Validation không cải thiện trong 10 epoch.

Việc shuffle trong DataLoader chỉ xáo thứ tự các window **bên trong tập Train**; không làm xáo trộn ranh giới thời gian Train/Validation/Test.

In [ ]:
def fit_model(model, X_train, y_train, X_val, y_val):
    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
        batch_size=BATCH_SIZE, shuffle=True
    )
    Xv, yv = torch.from_numpy(X_val).to(DEVICE), torch.from_numpy(y_val).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.MSELoss()

    best_loss, best_state, best_epoch, wait = float("inf"), None, 0, 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_losses = []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xv), yv).item()

        history.append({
            "epoch": epoch,
            "train_mse_scaled": float(np.mean(train_losses)),
            "validation_mse_scaled": val_loss
        })

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            print("Early stopping at epoch", epoch)
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch

def predict_scaled(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.from_numpy(X).to(DEVICE)).cpu().numpy().reshape(-1)


## 6. Huấn luyện từng chuỗi, lưu dự báo và tính metrics

Mỗi chuỗi được huấn luyện độc lập. Kết quả Validation và Test được đưa về đơn vị gốc trước khi tính MAE, RMSE, MASE.

In [ ]:
def run_one_series(customer):
    # Scaler fit duy nhất trên Train
    scaler = StandardScaler()
    train_raw = train_df[customer].to_numpy(dtype=np.float64)
    scaler.fit(train_raw.reshape(-1, 1))

    train_scaled = scaler.transform(train_raw.reshape(-1, 1)).reshape(-1).astype(np.float32)
    val_raw = val_df[customer].to_numpy(dtype=np.float64)
    test_raw = test_df[customer].to_numpy(dtype=np.float64)
    val_scaled = scaler.transform(val_raw.reshape(-1, 1)).reshape(-1).astype(np.float32)
    test_scaled = scaler.transform(test_raw.reshape(-1, 1)).reshape(-1).astype(np.float32)

    X_train, y_train = make_windows(train_scaled)
    X_val, y_val = make_eval_windows(train_scaled, val_scaled)

    model = ForecastModel().to(DEVICE)
    model, history, best_epoch = fit_model(model, X_train, y_train, X_val, y_val)

    val_pred = scaler.inverse_transform(
        predict_scaled(model, X_val).reshape(-1, 1)
    ).reshape(-1)
    test_X, _ = make_eval_windows(val_scaled, test_scaled)
    test_pred = scaler.inverse_transform(
        predict_scaled(model, test_X).reshape(-1, 1)
    ).reshape(-1)

    d_train = calculate_d_train(train_raw)
    val_metrics = evaluate_forecast(val_raw, val_pred, d_train)
    test_metrics = evaluate_forecast(test_raw, test_pred, d_train)

    # Checkpoint model + thông tin scaler
    torch.save({
        "model_name": "MLP",
        "customer": customer,
        "lookback": LOOKBACK,
        "best_epoch": best_epoch,
        "model_state_dict": model.state_dict(),
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
        "seed": SEED
    }, MODEL_DIR / f"MLP_{customer}.pt")

    history.to_csv(TABLE_DIR / f"history_{customer}.csv", index=False)
    for split_name, timestamps, actual, pred in [
        ("validation", val_df.index, val_raw, val_pred),
        ("test", test_df.index, test_raw, test_pred)
    ]:
        pd.DataFrame({
            "timestamp": timestamps,
            "customer": customer,
            "model": "MLP",
            "y_true": actual,
            "y_pred": pred,
            "error": actual - pred
        }).to_csv(PRED_DIR / f"{split_name}_{customer}.csv", index=False)

    # Learning curve
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history.epoch, history.train_mse_scaled, label="Train MSE")
    ax.plot(history.epoch, history.validation_mse_scaled, label="Validation MSE")
    ax.set(title=f"MLP learning curve - {customer}", xlabel="Epoch", ylabel="MSE (scaled)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"learning_curve_{customer}.png", dpi=150)
    plt.show()

    # Forecast vs actual: first 7 days of Test
    n = min(168, len(test_raw))
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(test_df.index[:n], test_raw[:n], label="Actual")
    ax.plot(test_df.index[:n], test_pred[:n], label="Prediction")
    ax.set(title=f"MLP Test forecast - {customer}", xlabel="Timestamp", ylabel="Hourly load")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"test_forecast_{customer}.png", dpi=150)
    plt.show()

    return {
        "customer": customer,
        "model": "MLP",
        "best_epoch": best_epoch,
        **{f"validation_{k}": v for k, v in val_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()}
    }


## 7. Chạy trên đủ 4 chuỗi

Cell này sẽ huấn luyện lần lượt MT_234, MT_217, MT_162, MT_314. Mỗi chuỗi tạo một model checkpoint và file dự báo riêng.

In [ ]:
results = []
for customer in CUSTOMERS:
    print(f"\n===== MLP | {customer} =====")
    results.append(run_one_series(customer))

results_df = pd.DataFrame(results)
results_df


## 8. Lưu bảng kết quả và kết luận baseline

Bảng Validation dùng để chọn mô hình/cấu hình. Test là đánh giá cuối cùng, không dùng để tinh chỉnh. Sau khi chạy cả notebook MLP và LSTM, ghép kết quả hai bảng với baseline truyền thống để so sánh.

In [ ]:
results_df.to_csv(TABLE_DIR / "metrics_4_series.csv", index=False)
results_df[[
    "customer", "model", "best_epoch",
    "validation_MAE", "validation_RMSE", "validation_MASE",
    "test_MAE", "test_RMSE", "test_MASE"
]]
